# HODGE v10a.29b — M5-First Blind \(m_5/m_6/m_7\) A100 Runner

This notebook is a runner/hardening layer for:

- `NB_O4_hodge_v10a26_factor52complete_exactsw_rootedoracle_a100.ipynb`
- `ENGINE_O4_hodge_v10a28_orderaware_gram_firewall_a100.py`

The original runner executed the entire monolithic v10a.26 code cell. That cell contains the completed one-face fourth-order preflight, the 33-shape fourth-order production loop, the rooted incidence transform, and the final M4 adjudication. This revision does **not** execute that suffix.

On a fresh kernel, the notebook executes only the shared v10a.26 source prefix needed to define the engine used by v10a.28, stopping before `shape_cache=_v26_load_checkpoint()`. It then installs the already-completed size-one rooted coefficient row through order four as a frozen comparator:

\[
(8/3,\;1,\;1/2,\;7/32,\;143/8960).
\]

The first requested-order frontier test is the **order-5 firewall**. The first production run is **blind M5**. No standalone order-4 v10a.28 invocation is permitted anywhere in this notebook.

Workflow:

1. import the common v10a.26 engine namespace while structurally excluding its M4 preflight/production suffix;
2. install the frozen one-face O0–O4 comparator as data rather than recomputing it;
3. run the v10a.28 order-5 firewall;
4. if every firewall gate passes, run blind order-5 production on the A100;
5. checkpoint each completed rooted shape and freeze the blind M5 result;
6. leave M6 and M7 locked until the preceding result is externally validated.

The M5 engine returns the coefficient vector through order five and checks lower-order entries produced inside the same M5 calculation. That is not a second M4 production run.

**Important:** production coefficients remain floating-point computer-assisted results until separately adjudicated by an exact rational backend.

In [ ]:
from __future__ import annotations

import os, sys, json, time, hashlib, platform, subprocess, shutil, zipfile
from pathlib import Path
from datetime import datetime, timezone

# ---------- USER SETTINGS ----------
USE_GOOGLE_DRIVE = True
WORKDIR_NAME = "HODGE_BLIND_M5_M7"
AUTO_UPLOAD_MISSING_FILES = True

V26_NAME = "NB_O4_hodge_v10a26_factor52complete_exactsw_rootedoracle_a100.ipynb"
V28_NAME = "ENGINE_O4_hodge_v10a28_orderaware_gram_firewall_a100.py"

# Hard execution policy: this runner never invokes v10a.28 at order four and
# never executes the v10a.26 M4 preflight/33-shape/rooted-transform suffix.
ALLOW_M4_EXECUTION = False

# Frozen size-one rooted v10a.26 row, imported as data rather than recomputed:
# c0=8/3, c1=1, c2=1/2, c3=7/32, c4=143/8960.
V26_ONE_FACE_PREFIX_EXACT = ("8/3", "1", "1/2", "7/32", "143/8960")

# M5 production defaults for an A100.
M5_MAX_NEW_SHAPES = 0        # 0 = unlimited in one invocation
M5_TIME_BUDGET_MINUTES = 0   # 0 = no between-shape budget
GPU_SW_MIN_DIM = 64
HERM_AUDIT_PAIRS = 24
DUPLICATE_CHECKS = 1
HEARTBEAT_SECONDS = 20

print("UTC:", datetime.now(timezone.utc).isoformat())
print("Python:", sys.version.replace("\n", " "))
print("Platform:", platform.platform())
print("Execution policy: v10a.26 M4 production suffix disabled; first requested order is M5.")

In [ ]:

# A100 / CUDA environment check. Installs only missing lightweight dependencies.
def ensure_import(import_name, pip_name=None):
    try:
        return __import__(import_name)
    except Exception:
        if pip_name is None:
            raise
        print(f"Installing missing package: {pip_name}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])
        return __import__(import_name)

np = ensure_import("numpy", "numpy")
sp = ensure_import("sympy", "sympy")
oe = ensure_import("opt_einsum", "opt_einsum")

try:
    import cupy as cp
except Exception:
    print("CuPy not importable; installing cupy-cuda12x.")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "cupy-cuda12x"])
    import cupy as cp

gpu_count = int(cp.cuda.runtime.getDeviceCount())
if gpu_count < 1:
    raise RuntimeError("No CUDA GPU detected. Select an A100 GPU runtime before continuing.")

props = cp.cuda.runtime.getDeviceProperties(0)
name = props["name"].decode() if isinstance(props["name"], (bytes, bytearray)) else str(props["name"])
free_b, total_b = cp.cuda.runtime.memGetInfo()

print("CUDA devices:", gpu_count)
print("GPU 0:", name)
print(f"GPU memory: free={free_b/2**30:.2f} GiB / total={total_b/2**30:.2f} GiB")
if "A100" not in name.upper():
    print("WARNING: runtime is not reporting an A100; the code can still run, but this notebook was tuned for A100.")


In [ ]:

# Mount Drive for durable checkpoints, or fall back to /content.
if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        WORKDIR = Path("/content/drive/MyDrive") / WORKDIR_NAME
    except Exception as exc:
        print("Drive mount unavailable; using /content:", exc)
        WORKDIR = Path("/content") / WORKDIR_NAME
else:
    WORKDIR = Path("/content") / WORKDIR_NAME

WORKDIR.mkdir(parents=True, exist_ok=True)
print("WORKDIR:", WORKDIR)


In [ ]:

# Locate or upload the two source artifacts.
CONTENT = Path("/content")

def locate(name: str) -> Path | None:
    candidates = [
        CONTENT / name,
        WORKDIR / name,
        Path.cwd() / name,
    ]
    for p in candidates:
        if p.exists():
            return p
    return None

def upload_if_missing(name: str) -> Path:
    p = locate(name)
    if p is not None:
        return p
    if not AUTO_UPLOAD_MISSING_FILES:
        raise FileNotFoundError(name)
    try:
        from google.colab import files
    except Exception as exc:
        raise FileNotFoundError(f"{name} not found and Colab upload is unavailable") from exc
    print(f"Upload: {name}")
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError(f"No file uploaded for {name}")
    # Accept exact file name first; otherwise accept a single uploaded file and rename it.
    if name in uploaded:
        p = CONTENT / name
    elif len(uploaded) == 1:
        src_name = next(iter(uploaded))
        src = CONTENT / src_name
        p = CONTENT / name
        if src != p:
            shutil.move(str(src), str(p))
    else:
        raise RuntimeError(f"Expected {name}; received {list(uploaded)}")
    return p

V26_PATH = upload_if_missing(V26_NAME)
V28_PATH = upload_if_missing(V28_NAME)

print("v10a.26:", V26_PATH)
print("v10a.28:", V28_PATH)


In [ ]:

# Provenance hashes. No requested-order target values are loaded.
def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()

V26_SHA256 = sha256_file(V26_PATH)
V28_SHA256 = sha256_file(V28_PATH)

print("v10a.26 SHA-256:", V26_SHA256)
print("v10a.28 SHA-256:", V28_SHA256)

src28 = V28_PATH.read_text(encoding="utf-8", errors="strict")
assert "target coefficient           : NOT LOADED" in src28
assert "external target remains absent" in src28
print("Blind-source declaration found in v10a.28.")


## Import the shared engine namespace without rerunning M4 production

The original loader executed every v10a.26 code cell wholesale. Because v10a.26 is monolithic, that also launched the completed M4 preflight and 33-shape rooted production oracle.

This loader instead:

- reads the v10a.26 source;
- locates the verified line `shape_cache=_v26_load_checkpoint();`;
- compiles and executes only the source before that line;
- verifies that the skipped suffix contains the one-face preflight, production loop, rooted incidence transform, and final fourth-order adjudication;
- refuses to fall back to full-cell execution if the source layout does not match.

The common engine/setup prefix is still required because v10a.28 is not standalone. The expensive v10a.26 M4 preflight, 33-shape calculation, rooted transform, and unblind are excluded. The completed one-face O0–O4 row is then inserted into `shape_cache` as regression data.

In [ ]:
import nbformat
from fractions import Fraction

V28_REQUIRED_GLOBALS = (
    "L", "N", "faces", "verts", "T1_POLS", "anchor_faces", "V23C_ROOT",
    "V23C_POL", "_FAST_EPS", "oe", "LXState", "_V17_VAC",
    "_v17_apply_W_faces", "_v17_apply_W_labeled", "_v17_connected",
    "_v17_phys_index", "_v17_translate_support", "_v17_translate_face",
    "_v23c_split_h0", "_v23c_rooted_connected_subsets", "_v24c_shape_key",
    "_v24c_candidate_supports", "_v10a3_face_state", "_v10a3_h0_state_inner",
    "_v10a3_physical_blocks", "_v10a3_compress_state", "_v9_flux_key_state",
    "_joint_canon_states", "lx_combine_bra_ket", "_v26_sw_blocks",
    "_v23_sw_exact", "_v23_sp", "_v23_random", "_V23CF",
    "_v26_singlet_multiplicity", "_V17_NEIGH", "_v23c_fit_cluster",
)

# Preferred cutoff first; the comment marker tolerates harmless formatting drift.
# Both occur after the shared definitions and before the M4 preflight/production loop.
V26_BOOTSTRAP_STOP_MARKERS = (
    "\nshape_cache=_v26_load_checkpoint();",
    "\n# Mandatory cold preflight on the literal one-face rooted cluster.",
)
V26_SKIPPED_SECTION_SENTINELS = (
    "Mandatory cold preflight on the literal one-face rooted cluster",
    "for ci,C in enumerate(CLUST,1)",
    "ROOTED INCIDENCE TRANSFORM",
    "FINAL FOURTH-ORDER UNBLIND",
)

# Defined even when the required namespace is already present in the kernel.
V26_BOOTSTRAP_EXECUTED = False
V26_BOOTSTRAP_PREFIX_SHA256 = None
V26_SKIPPED_SUFFIX_SHA256 = None
V26_BOOTSTRAP_MARKER = "required v10a.26 namespace already present; source bootstrap skipped"


def missing_base_symbols():
    return [name for name in V28_REQUIRED_GLOBALS if name not in globals()]


def _v26_source(path: Path) -> str:
    doc = nbformat.read(path, as_version=4)
    code = [cell.source for cell in doc.cells if cell.cell_type == "code" and cell.source.strip()]
    if not code:
        raise RuntimeError("v10a.26 contains no executable code cell")
    # The certified source currently has one code cell. Joining also handles a later split.
    return "\n\n".join(code)


def _cut_v26_before_m4(source: str) -> tuple[str, str, str]:
    hits = [(source.find(marker), marker) for marker in V26_BOOTSTRAP_STOP_MARKERS]
    hits = [(pos, marker) for pos, marker in hits if pos >= 0]
    if not hits:
        raise RuntimeError(
            "Refusing to execute v10a.26: the verified pre-M4 cutoff marker was not found. "
            "This runner has no full-cell fallback."
        )

    cut, marker = min(hits, key=lambda item: item[0])
    prefix, skipped = source[:cut], source[cut:]

    absent = [token for token in V26_SKIPPED_SECTION_SENTINELS if token not in skipped]
    if absent:
        raise RuntimeError(
            "Refusing to execute v10a.26: cutoff did not isolate the expected M4 suffix; "
            "missing skipped-section sentinels: " + ", ".join(absent)
        )

    forbidden_in_prefix = [
        token for token in V26_SKIPPED_SECTION_SENTINELS
        if token in prefix
    ]
    if forbidden_in_prefix:
        raise RuntimeError(
            "Internal cutoff failure: M4 production text remained executable: "
            + ", ".join(forbidden_in_prefix)
        )

    compile(prefix, str(V26_PATH) + "::<shared-prefix-only>", "exec")
    return prefix, skipped, marker


def execute_v26_shared_prefix(path: Path):
    if ALLOW_M4_EXECUTION:
        raise RuntimeError("ALLOW_M4_EXECUTION must remain False in the M5-first runner")

    source = _v26_source(path)
    prefix, skipped, marker = _cut_v26_before_m4(source)

    # Defense in depth. The production driver is physically absent from `prefix`;
    # these values also prevent accidental use by any future code added earlier.
    os.environ["V10A26_PREFLIGHT_ONLY"] = "1"
    os.environ["V10A26_RESUME"] = "0"
    os.environ["V10A26_CHECKPOINT"] = str(WORKDIR / "M4_EXECUTION_DISABLED.pkl")

    global V26_BOOTSTRAP_EXECUTED
    global V26_BOOTSTRAP_PREFIX_SHA256, V26_SKIPPED_SUFFIX_SHA256, V26_BOOTSTRAP_MARKER
    V26_BOOTSTRAP_EXECUTED = True
    V26_BOOTSTRAP_PREFIX_SHA256 = hashlib.sha256(prefix.encode("utf-8")).hexdigest()
    V26_SKIPPED_SUFFIX_SHA256 = hashlib.sha256(skipped.encode("utf-8")).hexdigest()
    V26_BOOTSTRAP_MARKER = marker.strip()

    print("Loading shared v10a.26 prefix; M4 preflight/production suffix is excluded.", flush=True)
    exec(compile(prefix, str(path) + "::<shared-prefix-only>", "exec"), globals())
    print("v10a.26 shared prefix executed.")
    print("  cutoff marker:", V26_BOOTSTRAP_MARKER)
    print("  executable-prefix SHA-256:", V26_BOOTSTRAP_PREFIX_SHA256)
    print("  skipped M4 suffix SHA-256:", V26_SKIPPED_SUFFIX_SHA256)


missing = missing_base_symbols()
if missing:
    print("Shared engine symbols missing:", len(missing))
    execute_v26_shared_prefix(V26_PATH)
    missing = missing_base_symbols()

if missing:
    raise RuntimeError("shared-prefix v10a.26 bootstrap did not create: " + ", ".join(missing))

# Install the completed size-one rooted row as immutable comparator data.
V26_ONE_FACE_PREFIX = np.array(
    [
        float(Fraction(8, 3)),
        float(Fraction(1, 1)),
        float(Fraction(1, 2)),
        float(Fraction(7, 32)),
        float(Fraction(143, 8960)),
    ],
    dtype=np.float64,
)
_one_face = frozenset((int(V23C_ROOT),))
_one_face_key = _v24c_shape_key(_one_face)

# Preserve an already-loaded cache rather than deleting completed in-memory data.
_existing_shape_cache = globals().get("shape_cache")
shape_cache = dict(_existing_shape_cache) if isinstance(_existing_shape_cache, dict) else {}
_existing_one_face = shape_cache.get(_one_face_key)
if isinstance(_existing_one_face, dict) and len(_existing_one_face.get("coef", ())) >= 5:
    _existing_prefix = np.asarray(_existing_one_face["coef"][:5], dtype=np.float64)
    _existing_error = float(np.max(np.abs(_existing_prefix - V26_ONE_FACE_PREFIX)))
    if _existing_error >= 3e-9:
        raise RuntimeError(
            "Preloaded one-face comparator disagrees with the completed v10a.26 row: "
            f"max error={_existing_error:.3e}"
        )
    print("Preserved matching preloaded one-face comparator.")
else:
    shape_cache[_one_face_key] = {
        "coef": V26_ONE_FACE_PREFIX.copy(),
        "source": "frozen completed v10a.26 size-one rooted row; M4 suffix not executed",
    }
    print("Installed frozen one-face O0-O4 comparator without recomputation.")

assert tuple(V26_ONE_FACE_PREFIX_EXACT) == ("8/3", "1", "1/2", "7/32", "143/8960")
assert np.all(np.isfinite(V26_ONE_FACE_PREFIX))

print("v10a.26 shared-namespace firewall: PASS")
print("v10a.26 M4 preflight/33-shape/rooted suffix: EXCLUDED")
print("Standalone v10a.28 order-4 execution: DISABLED")
print("Frozen one-face comparator:", V26_ONE_FACE_PREFIX_EXACT)
print("L =", L, "N =", N)

## M5 firewall — first requested-order frontier test

This is the first requested-order test launched by the notebook:

- requested order: **5**;
- Haar occurrence cap: 7;
- one-face order-5 physical/Krylov calculation;
- no support census;
- no production shapes;
- no order-4 v10a.28 invocation.

The O0–O4 comparison uses the frozen completed v10a.26 size-one row installed above. It does not rebuild that row.

In [ ]:
def configure_v28(order: int, mode: str, run_census: bool, production: bool):
    order = int(order)
    if order not in (5, 6, 7):
        raise ValueError("M5-first runner permits only orders 5, 6, and 7")
    cap = 7 if order == 5 else 9

    os.environ["V28_ORDER"] = str(order)
    os.environ["V28_MODE"] = mode
    os.environ["V28_HAAR_CAP"] = str(cap)
    os.environ["V28_RUN_CENSUS"] = "1" if run_census else "0"
    os.environ["V28_GPU"] = "1"
    os.environ["V28_GPU_SW_MIN_DIM"] = str(GPU_SW_MIN_DIM)
    os.environ["V28_HERMITICITY_AUDIT_PAIRS"] = str(HERM_AUDIT_PAIRS)
    os.environ["V28_DUPLICATE_CHECKS"] = str(DUPLICATE_CHECKS)
    os.environ["V28_HEARTBEAT"] = str(HEARTBEAT_SECONDS)
    os.environ["V28_RESUME"] = "1"
    # The comparator is already installed from the completed run. Never rebuild it.
    os.environ["V28_ALLOW_REFERENCE_REBUILD"] = "0"

    order_dir = WORKDIR / f"m{order}"
    order_dir.mkdir(parents=True, exist_ok=True)
    os.environ["V28_CHECKPOINT"] = str(order_dir / "shapes.pkl")
    os.environ["V28_CENSUS_CHECKPOINT"] = str(order_dir / "census.pkl")

    if production:
        os.environ["V28_PRODUCTION_CONFIRM"] = f"YES_ORDER_{order}"
        if order == 5:
            os.environ["V28_MAX_NEW_SHAPES"] = str(M5_MAX_NEW_SHAPES)
            os.environ["V28_TIME_BUDGET_MINUTES"] = str(M5_TIME_BUDGET_MINUTES)
        else:
            # Higher-order default: one atomic shape per invocation until tuned.
            os.environ["V28_MAX_NEW_SHAPES"] = "1"
            os.environ["V28_TIME_BUDGET_MINUTES"] = "30"
    else:
        os.environ["V28_PRODUCTION_CONFIRM"] = ""
        os.environ["V28_MAX_NEW_SHAPES"] = "1"
        os.environ["V28_TIME_BUDGET_MINUTES"] = "30"


def exec_v28():
    requested = int(os.environ.get("V28_ORDER", "-1"))
    if requested == 4:
        raise RuntimeError("Order-4 execution is disabled in this notebook")
    if requested not in (5, 6, 7):
        raise RuntimeError(f"Unexpected V28_ORDER={requested}")
    code = V28_PATH.read_text(encoding="utf-8")
    exec(compile(code, str(V28_PATH), "exec"), globals())


configure_v28(order=5, mode="firewall", run_census=False, production=False)
assert os.environ["V28_ORDER"] == "5"
exec_v28()

if not all(ok for _, ok, _ in V28_GATES):
    raise RuntimeError("ORDER-5 FIREWALL FAILED. Do not start production.")

print("\nORDER-5 FIREWALL: PASS")
print("No v10a.26 M4 preflight/33-shape production suffix was executed.")

## Blind M5 production

This is the first requested-order production cell. It runs only with `V28_ORDER=5`.

The order-5 SW/BCH calculation produces the coefficient vector through \(m_5\), so the final rooted transform also checks the known lower coefficients. Those values come from the M5 calculation itself; this cell does not launch a separate M4 production calculation.

With `M5_MAX_NEW_SHAPES=0` and `M5_TIME_BUDGET_MINUTES=0`, the cell attempts all remaining M5 shapes in one invocation. If Colab disconnects, rerun this cell; it resumes from `m5/shapes.pkl`.

In [ ]:

configure_v28(order=5, mode="production", run_census=True, production=True)
t_m5 = time.time()
exec_v28()
elapsed_m5 = time.time() - t_m5

print(f"\nOrder-5 invocation elapsed: {elapsed_m5/3600:.3f} h")

# V28_RESULT is either a complete result or an intentional budget-stop payload.
if V28_RESULT is None:
    raise RuntimeError("v10a.28 did not return V28_RESULT")

print("V28_RESULT complete:", V28_RESULT.get("complete", True))
if not V28_RESULT.get("complete", True):
    print("Checkpoint saved. Rerun this production cell to continue.")


In [ ]:
# Freeze a compact blind M5 provenance record when production is complete.
def jsonable(x):
    if isinstance(x, np.ndarray):
        return x.tolist()
    if isinstance(x, (np.floating, np.integer)):
        return x.item()
    if isinstance(x, Path):
        return str(x)
    if isinstance(x, dict):
        return {str(k): jsonable(v) for k, v in x.items()}
    if isinstance(x, (list, tuple)):
        return [jsonable(v) for v in x]
    return x

M5_DIR = WORKDIR / "m5"
M5_SUMMARY = M5_DIR / "blind_m5_summary.json"

if V28_RESULT.get("complete", True):
    coeff = np.asarray(V28_RESULT["coefficients"], dtype=float)
    payload = {
        "schema": "hodge-v10a29b-m5-first-freeze-v2",
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "order": 5,
        "blind": True,
        "external_requested_order_target_loaded": False,
        "v10a26_m4_preflight_executed_by_runner": False,
        "v10a26_m4_33_shape_production_executed_by_runner": False,
        "v10a26_m4_rooted_transform_executed_by_runner": False,
        "standalone_v10a28_order4_invocation": False,
        "v26_bootstrap_executed": bool(V26_BOOTSTRAP_EXECUTED),
        "v26_bootstrap_mode": "shared source prefix ending before shape_cache/M4 preflight/production",
        "v26_bootstrap_cutoff_marker": V26_BOOTSTRAP_MARKER,
        "v26_bootstrap_prefix_sha256": V26_BOOTSTRAP_PREFIX_SHA256,
        "v26_skipped_m4_suffix_sha256": V26_SKIPPED_SUFFIX_SHA256,
        "v26_one_face_comparator_exact": list(V26_ONE_FACE_PREFIX_EXACT),
        "coefficient_vector_m0_to_m5": coeff.tolist(),
        "m5": float(coeff[5]),
        "v10a26_sha256": V26_SHA256,
        "v10a28_sha256": V28_SHA256,
        "v28_schema": V28_RESULT.get("schema"),
        "v28_signature": V28_RESULT.get("signature"),
        "concrete_clusters": int(V28_RESULT.get("concrete_clusters", 0)),
        "shape_classes": int(V28_RESULT.get("shapes", 0)),
        "haar_cap": int(V28_RESULT.get("haar_cap", 0)),
        "krylov_depth": int(V28_RESULT.get("krylov_depth", 0)),
        "gates": [
            {"name": n, "passed": bool(ok), "detail": d}
            for n, ok, d in V28_GATES
        ],
        "gpu": name,
        "elapsed_seconds_last_invocation": float(elapsed_m5),
        "checkpoint": os.environ["V28_CHECKPOINT"],
        "census_checkpoint": os.environ["V28_CENSUS_CHECKPOINT"],
    }
    M5_SUMMARY.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    print("FROZEN BLIND m5:", repr(float(coeff[5])))
    print("Summary:", M5_SUMMARY)
    print("Summary SHA-256:", sha256_file(M5_SUMMARY))
else:
    print("M5 is not complete yet; no blind coefficient was frozen.")


## External holdout boundary

At this point, if `blind_m5_summary.json` exists:

1. copy it off the runtime;
2. record its SHA-256;
3. compare \(m_5\) to the external published holdout **outside this notebook**;
4. if and only if you accept the result, create the validation flag below.

The published target is intentionally absent here.


In [ ]:

# Run this only after external validation of the frozen blind m5 result.
MARK_M5_EXTERNALLY_VALIDATED = False

flag5 = WORKDIR / "m5" / "M5_EXTERNALLY_VALIDATED.flag"
if MARK_M5_EXTERNALLY_VALIDATED:
    if not M5_SUMMARY.exists():
        raise RuntimeError("Cannot validate: blind_m5_summary.json is missing.")
    flag5.write_text(
        "User marked frozen blind m5 externally validated at "
        + datetime.now(timezone.utc).isoformat() + "\n",
        encoding="utf-8",
    )
    print("Created:", flag5)
else:
    print("m6 remains locked. Set MARK_M5_EXTERNALLY_VALIDATED=True only after external comparison.")


## \(m_6\) and \(m_7\) — deliberately locked

The engine supports orders 6 and 7 with Haar cap 9 and Krylov depth 3.

This notebook never permits a standalone v10a.28 order-4 run. It also does not automatically proceed past M5, preventing a failed M5 result from contaminating the next stage.

For M6, create the `M5_EXTERNALLY_VALIDATED.flag` above.

For M7, first freeze and externally validate M6, then create `M6_EXTERNALLY_VALIDATED.flag`.

Higher-order runs default to one new atomic shape per invocation. The limit can be raised later without changing the physics.

In [ ]:

def run_higher_order(order: int):
    order = int(order)
    if order not in (6, 7):
        raise ValueError("This helper is only for m6 or m7.")

    if not (WORKDIR / "m5" / "M5_EXTERNALLY_VALIDATED.flag").exists():
        raise RuntimeError("m6/m7 locked: m5 has not been marked externally validated.")
    if order == 7 and not (WORKDIR / "m6" / "M6_EXTERNALLY_VALIDATED.flag").exists():
        raise RuntimeError("m7 locked: m6 has not been marked externally validated.")

    # Short firewall at the requested order.
    configure_v28(order=order, mode="firewall", run_census=False, production=False)
    exec_v28()
    if not all(ok for _, ok, _ in V28_GATES):
        raise RuntimeError(f"ORDER-{order} FIREWALL FAILED.")

    # Production, resumable. Defaults to one new shape/invocation.
    configure_v28(order=order, mode="production", run_census=True, production=True)
    t0 = time.time()
    exec_v28()
    dt = time.time() - t0
    print(f"Order-{order} invocation elapsed: {dt/3600:.3f} h")

    if not V28_RESULT.get("complete", True):
        print("Incomplete by intentional budget/shape limit. Rerun run_higher_order(order).")
        return V28_RESULT

    coeff = np.asarray(V28_RESULT["coefficients"], dtype=float)
    order_dir = WORKDIR / f"m{order}"
    summary = order_dir / f"blind_m{order}_summary.json"

    # Self-generated lower-order consistency locks.
    lower_checks = {}
    m5_data = json.loads((WORKDIR / "m5" / "blind_m5_summary.json").read_text())
    lower_checks["m5_vs_frozen_m5"] = {
        "current": float(coeff[5]),
        "frozen": float(m5_data["m5"]),
        "abs_error": abs(float(coeff[5]) - float(m5_data["m5"])),
    }
    if order == 7:
        m6_data = json.loads((WORKDIR / "m6" / "blind_m6_summary.json").read_text())
        lower_checks["m6_vs_frozen_m6"] = {
            "current": float(coeff[6]),
            "frozen": float(m6_data["m6"]),
            "abs_error": abs(float(coeff[6]) - float(m6_data["m6"])),
        }

    payload = {
        "schema": f"hodge-v10a29-blind-m{order}-freeze-v1",
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "order": order,
        "blind": True,
        "external_requested_order_target_loaded": False,
        f"m{order}": float(coeff[order]),
        "coefficient_vector": coeff.tolist(),
        "lower_order_self_consistency": lower_checks,
        "v10a26_sha256": V26_SHA256,
        "v10a28_sha256": V28_SHA256,
        "v28_schema": V28_RESULT.get("schema"),
        "v28_signature": V28_RESULT.get("signature"),
        "gates": [{"name": n, "passed": bool(ok), "detail": d} for n, ok, d in V28_GATES],
        "gpu": name,
    }
    summary.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    print(f"FROZEN BLIND m{order}:", repr(float(coeff[order])))
    print("Summary:", summary)
    print("Summary SHA-256:", sha256_file(summary))
    return V28_RESULT

# Examples — leave commented until the validation flags exist:
# run_higher_order(6)
# run_higher_order(7)


In [ ]:

# Package compact provenance/results. Checkpoints are intentionally excluded because they can be very large.
bundle = WORKDIR / "blind_results_compact.zip"
with zipfile.ZipFile(bundle, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for p in sorted(WORKDIR.rglob("*")):
        if not p.is_file():
            continue
        if p.suffix == ".pkl":
            continue
        z.write(p, p.relative_to(WORKDIR))
    # Include exact source files used.
    z.write(V26_PATH, Path("sources") / V26_PATH.name)
    z.write(V28_PATH, Path("sources") / V28_PATH.name)

print("Compact bundle:", bundle)
print("SHA-256:", sha256_file(bundle))


## What to send back after the A100 run

For M5, send:

- the final order-5 firewall summary;
- `blind_m5_summary.json`;
- the last 100–200 lines around `ROOTED INCIDENCE TRANSFORM`;
- any exception traceback;
- if the run stops intentionally, the `completed_shapes / total_shapes` count and the last completed shape dimensions.

The startup output should explicitly show:

- `v10a.26 M4 preflight/33-shape/rooted suffix: EXCLUDED`;
- `Standalone v10a.28 order-4 execution: DISABLED`;
- `ORDER-5 FIREWALL` as the first requested-order frontier test;
- no v10a.26 `shape DONE ... 33/33` production sequence.

Do not paste the external M5 target into the runtime before the blind summary is frozen.